# **Proyecto Etapa 3 — Aprendizaje Supervisado y No Supervisado con PySpark**

### **Curso: TC5057 · Análisis de Grandes Volúmenes de Datos**
#### **Tecnológico de Monterrey**
##### **Profesor Titular: Dr. Iván Olmos Pineda**

---

### **Equipo #17**
#### **Tutor: José Carlos Soto**

| Nombre | Matrícula |
|--------|-----------|
| Diego Falcón Costilla | A00000000 |

---

**Dataset:** GTEx Analysis V10 — Gene Expression TPM (NIH / Broad Institute)  
**Archivo:** `GTEx_Analysis_2022-06-06_v10_RNASeQCv2.4.2_gene_tpm_non_lcm.gct`  
**Genes:** 59,033 | **Muestras RNASEQ:** 19,788 | **Donantes:** 981

---
## 1. Construcción de la muestra M

### 1.1 Definición de M

La muestra M se define como el conjunto de **todas** las muestras RNASEQ disponibles en GTEx V10, organizadas en las **10 particiones** derivadas de las variables de caracterización:

$$M = \{M_i : M_i \text{ es una partición de TISSUE\_GROUP} \times \text{SEX\_LABEL}\},\quad i = 1, \ldots, 10$$

con 5 grupos de tejido × 2 sexos biológicos como ejes de particionamiento.

### 1.2 Evaluación de representatividad y mejoras implementadas

Se evaluó si la muestra M es suficientemente representativa de la población P (donantes adultos del proyecto GTEx V10 con datos RNASEQ). El análisis identificó los siguientes ajustes necesarios respecto a versiones previas de trabajo:

| Aspecto evaluado | Versión previa | Versión actual | Justificación |
|-----------------|----------------|----------------|---------------|
| **Cobertura de muestras** | Sub-muestra reducida (1/100 de columnas) | Todas las 19,788 muestras RNASEQ | La tarea exige trabajar con la muestra lo más completa posible |
| **Selección de genes** | Sub-selección aleatoria de columnas | Top-500 genes por varianza inter-muestras | Los genes de mayor varianza son los más discriminantes biológicamente (Law et al., 2016) |
| **División train/test** | Aleatoria por muestra (sesgo intra-donante) | Por donante (`SUBJID`) — garantiza `Tri ∩ Tsi = ∅` | Evita que el modelo aprenda perfiles individuales en lugar de patrones generalizables |
| **Cobertura de particiones** | 2–4 particiones | 10 particiones completas | M debe representar toda la diversidad tisular de P |
| **SMTSD disponible** | No incluido | Incluido en metadatos | Permite clasificación fina por sub-tipo de tejido si se requiere |

**Conclusión de representatividad:** la muestra M con 19,616 muestras cubre los 946 donantes únicos disponibles, representa los 5 grupos de tejido y ambos sexos biológicos, y selecciona los genes de mayor varianza para maximizar la señal discriminante. Se considera suficientemente representativa para el análisis de aprendizaje automático de esta etapa.

### 1.3 Estrategia de carga del dataset

El archivo TPM tiene **59,033 genes × 19,788 muestras** (~4.7 GB). Se aplica una estrategia en dos pasos para mantener factibilidad computacional sin perder representatividad biológica:

1. **Cálculo de varianza por bloques:** lectura chunked (500 genes/bloque) con `pandas.read_csv`, ~80 MB pico por bloque. Varianza calculada de forma vectorizada con `DataFrame.var(axis=1)`.
2. **Carga de la matriz final:** solo las 500 filas de los top genes, para todas las 19,616 muestras → 78.5 MB en memoria.

Los metadatos (particiones, donantes, sexo) se cargan y procesan con **PySpark** para aprovechar el procesamiento distribuido en la unión y filtrado de los 19,788 registros.

In [1]:
import sys, os

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

_conda_env = os.path.dirname(sys.executable)
_java_home = os.path.join(_conda_env, 'Library', 'lib', 'jvm')
if os.path.isdir(_java_home):
    os.environ['JAVA_HOME'] = _java_home

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType

sys.path.insert(0, os.path.abspath('../src'))
from GlobalVariables import (
    FILE_PATH, SAMPLE_ATTRS_PATH, SUBJECT_PHENO_PATH,
    N_GENES, RANDOM_SEED
)

import random
import numpy as np
import pandas as pd
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'JAVA_HOME: {os.environ.get("JAVA_HOME", "ERROR")}')
print(f'Semilla aleatoria: {RANDOM_SEED}')
print(f'Genes en el dataset: {N_GENES:,}')

JAVA_HOME: C:\Users\diego\anaconda3\envs\big-data\Library\lib\jvm
Semilla aleatoria: 42
Genes en el dataset: 59,033


In [2]:
spark = SparkSession.builder \
    .master('local[*]') \
    .appName('GTEx_Etapa3_TC5057') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')
print(f'Workers: {spark.sparkContext.defaultParallelism}')

Spark version: 4.1.1
Workers: 32


In [3]:
# --- Carga de metadatos: todos los atributos de las muestras RNASEQ ---
sa_df = spark.read.csv(SAMPLE_ATTRS_PATH, sep='\t', header=True) \
    .select('SAMPID', 'SMTS', 'SMTSD', 'SMAFRZE') \
    .filter(F.col('SMAFRZE') == 'RNASEQ') \
    .withColumn('SUBJID', F.regexp_extract(F.col('SAMPID'), r'^(GTEX-[^-]+)', 1))

sp_df = spark.read.csv(SUBJECT_PHENO_PATH, sep='\t', header=True) \
    .select('SUBJID', 'SEX')

# Mapeo de SMTS a grupos de tejido (misma nomenclatura que Etapa 2)
tissue_group_col = (
    F.when(F.col('SMTS').isin('Brain', 'Nerve'), 'Nervioso')
     .when(F.col('SMTS').isin('Blood', 'Bone Marrow', 'Spleen'), 'Hematopoyetico')
     .when(F.col('SMTS').isin('Heart', 'Blood Vessel'), 'Cardiovascular')
     .when(F.col('SMTS').isin('Muscle', 'Adipose Tissue', 'Skin'), 'Musculoesqueletico')
     .otherwise('Visceral_Metabolico')
)
sex_label_col = F.when(F.col('SEX') == '1', 'Masculino').otherwise('Femenino')

meta_df = sa_df.join(sp_df, on='SUBJID', how='inner') \
    .withColumn('TISSUE_GROUP', tissue_group_col) \
    .withColumn('SEX_LABEL', sex_label_col) \
    .withColumn('COL_NAME',
        F.regexp_replace(F.regexp_replace(F.col('SAMPID'), '-', '_'), '\\.', '_'))

total_samples = meta_df.count()
total_donors  = meta_df.select('SUBJID').distinct().count()
print(f'Muestras RNASEQ en M: {total_samples:,}')
print(f'Donantes únicos     : {total_donors:,}')

Muestras RNASEQ en M: 19,788
Donantes únicos     : 946


In [4]:
# --- Distribución de las 10 particiones de M ---
print('Distribución de M por partición (TISSUE_GROUP × SEX_LABEL):')
partition_dist = meta_df.groupBy('TISSUE_GROUP', 'SEX_LABEL') \
    .count() \
    .orderBy('TISSUE_GROUP', 'SEX_LABEL')
partition_dist.show(20)

print('Distribución por grupo de tejido (total):')
meta_df.groupBy('TISSUE_GROUP').count().orderBy('count', ascending=False).show()

Distribución de M por partición (TISSUE_GROUP × SEX_LABEL):


+-------------------+---------+-----+
|       TISSUE_GROUP|SEX_LABEL|count|
+-------------------+---------+-----+
|     Cardiovascular| Femenino|  771|
|     Cardiovascular|Masculino| 1573|
|     Hematopoyetico| Femenino|  475|
|     Hematopoyetico|Masculino|  932|
| Musculoesqueletico| Femenino| 1349|
| Musculoesqueletico|Masculino| 2827|
|           Nervioso| Femenino| 1066|
|           Nervioso|Masculino| 2838|
|Visceral_Metabolico| Femenino| 2864|
|Visceral_Metabolico|Masculino| 5093|
+-------------------+---------+-----+

Distribución por grupo de tejido (total):


+-------------------+-----+
|       TISSUE_GROUP|count|
+-------------------+-----+
|Visceral_Metabolico| 7957|
| Musculoesqueletico| 4176|
|           Nervioso| 3904|
|     Cardiovascular| 2344|
|     Hematopoyetico| 1407|
+-------------------+-----+



### 1.2 Carga del dataset TPM y selección de genes por varianza

El archivo TPM tiene **59,033 genes (filas) × 19,788 muestras (columnas)**. Cargar la matriz completa requeriría ~4.7 GB en memoria. Para mantener factibilidad computacional sin perder representatividad biológica, se aplica una estrategia en dos pasos:

1. **Cálculo de varianza** por gen a través de todas las muestras válidas: lectura por bloques (`chunksize=500` genes) para mantener el pico de memoria en ~80 MB/bloque.
2. **Carga de la matriz final**: solo las filas correspondientes a los top-500 genes, para todas las 19,788 muestras → ~75 MB en memoria.

La selección por varianza está validada en la literatura de RNA-seq (Law et al., 2016) y en nuestros propios experimentos de Tarea 3: los top-500 genes por varianza contienen los marcadores tisulares más discriminantes (PLN, ACTN2, MYL7, ACTC1, etc.).

In [5]:
# --- Paso 1: obtener nombres de columna del archivo TPM ---
# Nota: el archivo GTEx usa guiones en los IDs de muestra (GTEX-1117F-...)
# pero Spark sanitiza a guiones bajos (GTEX_1117F_...). Se construye un mapeo bidireccional.
import pandas as pd

peek = pd.read_csv(FILE_PATH, sep='\t', skiprows=2, nrows=0)
all_file_cols = peek.columns.tolist()

# Mapeo: nombre_en_archivo (guiones) -> nombre_sanitizado (guiones bajos)
file_to_san = {
    c: c.replace('-', '_').replace('.', '_')
    for c in all_file_cols
    if c not in ('Name', 'Description')
}

meta_col_names = set(
    row['COL_NAME'] for row in meta_df.select('COL_NAME').collect()
)

# Columnas válidas: aquellas cuyo nombre sanitizado está en los metadatos
valid_sample_cols_file = [c for c, san in file_to_san.items() if san in meta_col_names]
rename_dash_to_san     = {c: file_to_san[c] for c in valid_sample_cols_file}

print(f'Columnas en el archivo TPM       : {len(all_file_cols):,}')
print(f'Columnas de muestra en el archivo: {len(file_to_san):,}')
print(f'Columnas en M (metadatos)        : {len(meta_col_names):,}')
print(f'Columnas válidas (intersección)  : {len(valid_sample_cols_file):,}')
print(f'Ejemplo mapeo: {list(rename_dash_to_san.items())[0]}')

Columnas en el archivo TPM       : 19,618
Columnas de muestra en el archivo: 19,616
Columnas en M (metadatos)        : 19,788
Columnas válidas (intersección)  : 19,616
Ejemplo mapeo: ('GTEX-1117F-0005-SM-HL9SH', 'GTEX_1117F_0005_SM_HL9SH')


In [6]:
# --- Paso 2: calcular varianza por gen (lectura por bloques) ---
# Cada bloque: 500 genes × ~19,616 muestras ≈ 80 MB pico en memoria
# Varianza calculada de forma vectorizada con DataFrame.var(axis=1) — evita iterrows()
CHUNK_SIZE = 500
gene_vars  = {}
usecols_file = ['Name'] + valid_sample_cols_file

n_chunks = -(-N_GENES // CHUNK_SIZE)  # ceil division
print(f'Calculando varianza para {N_GENES:,} genes en ~{n_chunks} bloques de {CHUNK_SIZE}...')

for i, chunk in enumerate(pd.read_csv(
        FILE_PATH, sep='\t', skiprows=2,
        chunksize=CHUNK_SIZE, usecols=usecols_file)):
    chunk = chunk.rename(columns=rename_dash_to_san).set_index('Name')
    chunk = chunk.apply(pd.to_numeric, errors='coerce').fillna(0.0)
    gene_vars.update(chunk.var(axis=1).to_dict())
    if (i + 1) % 20 == 0:
        print(f'  Bloque {i+1:>3} procesado ({(i+1)*CHUNK_SIZE:,} genes)')

gene_var_series = pd.Series(gene_vars).sort_values(ascending=False)
print(f'\nVarianza calculada para {len(gene_var_series):,} genes.')
print('Top 10 genes por varianza:')
print(gene_var_series.head(10))

Calculando varianza para 59,033 genes en ~119 bloques de 500...


  Bloque  20 procesado (10,000 genes)


  Bloque  40 procesado (20,000 genes)


  Bloque  60 procesado (30,000 genes)


  Bloque  80 procesado (40,000 genes)


  Bloque 100 procesado (50,000 genes)



Varianza calculada para 59,033 genes.
Top 10 genes por varianza:
ENSG00000244734.4     3.018102e+09
ENSG00000210082.2     6.927912e+08
ENSG00000198804.2     6.029021e+08
ENSG00000198712.1     4.503598e+08
ENSG00000198938.2     4.438837e+08
ENSG00000188536.13    4.395546e+08
ENSG00000198886.2     3.579125e+08
ENSG00000198899.2     3.578467e+08
ENSG00000275896.7     2.180929e+08
ENSG00000163220.11    2.052547e+08
dtype: float64


In [7]:
# --- Paso 3: seleccionar top-500 genes y cargar su matriz completa ---
TOP_N_GENES = 500
top_gene_ids = gene_var_series.head(TOP_N_GENES).index.tolist()

target_genes = set(top_gene_ids)
collected    = []

print(f'Cargando {TOP_N_GENES} genes seleccionados x {len(valid_sample_cols_file):,} muestras...')

for chunk in pd.read_csv(
        FILE_PATH, sep='\t', skiprows=2,
        chunksize=CHUNK_SIZE, usecols=usecols_file):
    chunk = chunk.rename(columns=rename_dash_to_san)
    hit = chunk[chunk['Name'].isin(target_genes)]
    if not hit.empty:
        collected.append(hit)

tpm_top = pd.concat(collected).set_index('Name')
tpm_top = tpm_top.apply(pd.to_numeric, errors='coerce').fillna(0.0)

print(f'Matriz de genes seleccionados: {tpm_top.shape[0]} genes × {tpm_top.shape[1]} muestras')
print(f'Tamaño en memoria: {tpm_top.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print(f'Ejemplo columna: {tpm_top.columns[0]}')

Cargando 500 genes seleccionados x 19,616 muestras...


Matriz de genes seleccionados: 500 genes × 19616 muestras


Tamaño en memoria: 78.5 MB
Ejemplo columna: GTEX_1117F_0005_SM_HL9SH


In [8]:
# --- Construcción de la matriz de features M: muestras × genes ---
# tpm_top index = gene IDs (Ensembl, con puntos)
# tpm_top columns = sample COL_NAMEs (ya en formato guiones bajos, desde cell-09)

# Sanitizar nombres de genes: reemplazar '.' por '_'
rename_genes = {g: g.replace('.', '_') for g in tpm_top.index}
tpm_top_san  = tpm_top.rename(index=rename_genes)

# Filtrar a los top genes que se cargaron
top_gene_ids_san = [g.replace('.', '_') for g in top_gene_ids if g in tpm_top.index]

# Transponer: filas=muestras, columnas=genes
tpm_T = tpm_top_san.loc[top_gene_ids_san].T.reset_index()
tpm_T = tpm_T.rename(columns={'index': 'COL_NAME'})
gene_cols = [c for c in tpm_T.columns if c != 'COL_NAME']

# Unir con metadatos (ambos usan formato guiones bajos en COL_NAME)
meta_pd = meta_df.select(
    'COL_NAME', 'TISSUE_GROUP', 'SEX_LABEL', 'SMTSD', 'SUBJID'
).toPandas()
M = tpm_T.merge(meta_pd, on='COL_NAME', how='inner')
M = M.dropna(subset=['TISSUE_GROUP', 'SUBJID'])
M[gene_cols] = M[gene_cols].fillna(0.0)

print(f'Muestra M final: {M.shape[0]:,} muestras × {len(gene_cols)} genes')
print(f'Donantes únicos en M: {M["SUBJID"].nunique():,}')
print('\nDistribución TISSUE_GROUP:')
print(M['TISSUE_GROUP'].value_counts())

Muestra M final: 19,616 muestras × 500 genes
Donantes únicos en M: 946

Distribución TISSUE_GROUP:
TISSUE_GROUP
Visceral_Metabolico    7785
Musculoesqueletico     4176
Nervioso               3904
Cardiovascular         2344
Hematopoyetico         1407
Name: count, dtype: int64


---
## 2. Construcción Train – Test

### 2.1 Estrategia de división

Para construir los conjuntos de entrenamiento y prueba a partir de M se adoptan los siguientes principios:

**División por donante (`GroupShuffleSplit` con `SUBJID`):**  
Cada muestra GTEx pertenece a un donante específico. El mismo donante puede contribuir muestras de múltiples tejidos. Una división aleatoria por muestra permite que el mismo donante aparezca en train y test simultáneamente, introduciendo un sesgo: el modelo aprende perfiles individuales de expresión génica en lugar de patrones generalizables. La corrección es dividir por `SUBJID`, garantizando `Tri ∩ Tsi = ∅` a nivel de donante.

**Variable objetivo — SMTSD (sub-tipo de tejido):**  
En lugar de los 5 grupos de tejido macro (`TISSUE_GROUP`), se clasifica directamente por `SMTSD`: ~50 sub-tipos de tejido (ventrículo izquierdo vs apéndice auricular, corteza cerebral vs amígdala, adiposo subcutáneo vs visceral, etc.). Este nivel de granularidad es biológicamente más exigente y produce un modelo genuinamente útil: la clasificación entre sub-tipos cercanos requiere aprender firmas de expresión génica sutiles, no solo diferencias macro entre tejidos.

**Proporción 80/20 y preservación de probabilidades de ocurrencia:**  
Con la división por donante y ~946 donantes únicos, la proporción de muestras de cada sub-tipo de tejido se preserva en ambos conjuntos — verificada explícitamente en la sección siguiente.

**Verificación formal:**  
- `Tri ∩ Tsi = ∅` — verificado por ausencia de donantes compartidos entre conjuntos.  
- `⋃ Tri ∪ Tsi = M` — verificado por suma de tamaños igual a |M|.  
- Proporciones por sub-tipo — verificadas comparando distribuciones en train y test.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

# Codificar SMTSD (sub-tipos de tejido — clasificación fina, ~50 clases)
le_tissue = LabelEncoder()
M = M.copy()
M = M.dropna(subset=['SMTSD'])
M['label'] = le_tissue.fit_transform(M['SMTSD'])

X = M[gene_cols].values
y = M['label'].values
groups = M['SUBJID'].values

n_classes = len(le_tissue.classes_)
print(f'Muestras tras filtrar SMTSD nulo: {len(M):,}')
print(f'Clases SMTSD: {n_classes} sub-tipos de tejido')
print()
for code_val, name in enumerate(le_tissue.classes_):
    cnt = (y == code_val).sum()
    print(f'  {code_val:>2}: {name} ({cnt:,})')

In [10]:
# --- División train/test por donante (80/20) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

train_donors = set(groups[train_idx])
test_donors  = set(groups[test_idx])
overlap      = train_donors & test_donors

print('=== Verificación de la división ===')
print(f'Train : {len(X_train):,} muestras | {len(train_donors):,} donantes')
print(f'Test  : {len(X_test):,} muestras  | {len(test_donors):,} donantes')
print(f'Total : {len(X_train)+len(X_test):,} == |M| = {len(M):,} → '
      f'{"OK" if len(X_train)+len(X_test)==len(M) else "ERROR"}')
print(f'Donantes en ambos conjuntos: {len(overlap)} → '
      f'{"OK (Tri ∩ Tsi = ∅)" if overlap==set() else "ERROR"}')

=== Verificación de la división ===
Train : 15,620 muestras | 756 donantes
Test  : 3,996 muestras  | 190 donantes
Total : 19,616 == |M| = 19,616 → OK
Donantes en ambos conjuntos: 0 → OK (Tri ∩ Tsi = ∅)


In [11]:
# --- Distribución por clase en cada conjunto (con proporciones) ---
print('Distribución TISSUE_GROUP en train vs test:')
print(f'  {"Clase":<25} {"Train N":>8} {"Train %":>9} {"Test N":>8} {"Test %":>9}')
print(f'  {"-"*65}')
for code_val, name in enumerate(le_tissue.classes_):
    n_tr  = (y_train == code_val).sum()
    n_ts  = (y_test  == code_val).sum()
    pct_tr = n_tr / len(y_train) * 100
    pct_ts = n_ts / len(y_test)  * 100
    print(f'  {name:<25} {n_tr:>8,} {pct_tr:>8.1f}% {n_ts:>8,} {pct_ts:>8.1f}%')
print(f'  {"-"*65}')
print(f'  {"Total":<25} {len(y_train):>8,} {"100.0%":>9} {len(y_test):>8,} {"100.0%":>9}')
print()
print('Las proporciones de cada clase se mantienen entre train y test.')
print('La división por donante no introduce sesgo en la probabilidad de ocurrencia de patrones.')

Distribución TISSUE_GROUP en train vs test:
  Clase                      Train N   Train %   Test N    Test %
  -----------------------------------------------------------------
  Cardiovascular               1,871     12.0%      473     11.8%
  Hematopoyetico               1,123      7.2%      284      7.1%
  Musculoesqueletico           3,311     21.2%      865     21.6%
  Nervioso                     3,096     19.8%      808     20.2%
  Visceral_Metabolico          6,219     39.8%    1,566     39.2%
  -----------------------------------------------------------------
  Total                       15,620    100.0%    3,996    100.0%

Las proporciones de cada clase se mantienen entre train y test.
La división por donante no introduce sesgo en la probabilidad de ocurrencia de patrones.


---
## 3. Selección de métricas para medir calidad de resultados

### 3.1 Consideraciones para grandes volúmenes de datos

Con 19,616 muestras y ~50 clases desbalanceadas (algunos sub-tipos de cerebro tienen 200+ muestras, algunos sub-tipos de tejido raro tienen <50), las métricas deben ser:

- **Robustas ante desbalance**: accuracy global es engañoso cuando las clases son desiguales. F1-macro pondera clases por igual, penalizando igual los errores en sub-tipos raros y en sub-tipos frecuentes.
- **Interpretables en contexto biológico**: precision y recall por clase revelan qué sub-tejidos son más difíciles de discriminar (ej. corteza prefrontal vs giro cingulado anterior).
- **Escalables**: PySpark `MulticlassClassificationEvaluator` calcula métricas directamente sobre DataFrames distribuidos.

### 3.2 Métricas para el modelo supervisado (Random Forest — clasificación multi-clase)

| Métrica | Fórmula | Justificación |
|---------|---------|---------------|
| **Accuracy** | (VP+VN)/(VP+VN+FP+FN) | Baseline rápido; referencia global |
| **F1-macro** | Promedio no ponderado de F1 por clase | Penaliza igualmente errores en sub-tipos raros y frecuentes |
| **Precision y Recall por clase** | TP/(TP+FP), TP/(TP+FN) | Revelan confusiones biológicas entre sub-tejidos cercanos |
| **Matriz de confusión** | N×N tabla de conteos | Diagnóstico visual de errores sistemáticos entre sub-tipos |

### 3.3 Métricas para el modelo no supervisado (K-Means — clustering)

| Métrica | Descripción | Justificación |
|---------|-------------|---------------|
| **Silhouette** (intrínseco) | Cohesión interna vs separación de clusters | No requiere etiquetas; mide calidad geométrica del clustering. Rango: [-1, 1] |
| **WCSS / Inercia** (intrínseco) | Suma de distancias cuadradas al centroide | Usado en el método del codo para seleccionar k óptimo |
| **Pureza** (extrínseco) | Fracción de la clase mayoritaria por cluster | Valida el clustering comparando con etiquetas de sub-tejido |

### 3.4 Implementación

Las métricas supervisadas se calculan con `MulticlassClassificationEvaluator` de PySpark MLlib y `classification_report` de scikit-learn. Las métricas de clustering se calculan con `silhouette_score` de scikit-learn aplicado post-hoc sobre predicciones recolectadas con `toPandas()` — estrategia que evita el bug de `ClusteringEvaluator` en Windows.

In [ ]:
# --- Definición de evaluadores ---
# RF supervisado: sklearn (misma decisión que StandardScaler/PCA — PySpark RF OOM en Windows con 54 clases)
# K-Means no supervisado: PySpark MLlib + sklearn silhouette post-hoc
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from sklearn.metrics import silhouette_score as sk_silhouette, classification_report
from sklearn.metrics import accuracy_score, f1_score as sk_f1

# Evaluadores PySpark — reservados para K-Means (el ClusteringEvaluator tiene bug en Windows;
# se usa sk_silhouette post-hoc igual que en Tarea 4)
kmeans_label_evaluator = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy')

print('Evaluadores definidos:')
print('  Supervisado RF : sklearn (accuracy_score, f1_score, classification_report)')
print('  No supervisado : sklearn.metrics.silhouette_score (post-hoc sobre predicciones)')

---
## 4. Entrenamiento de Modelos de Aprendizaje

### 4.1 Estrategia general

Se entrenan dos tipos de modelos complementarios:

| Modelo | Algoritmo | Implementación | Objetivo |
|--------|-----------|----------------|----------|
| **Supervisado** | Random Forest (100 árboles) | PySpark MLlib | Clasificar SMTSD (~50 sub-tipos de tejido) |
| **No supervisado** | K-Means (k óptimo por método del codo) | PySpark MLlib + sklearn PCA | Descubrir estructura latente en M |

**Variable objetivo — SMTSD vs TISSUE_GROUP:**  
Se clasifica por `SMTSD` (sub-tipo de tejido) en lugar de `TISSUE_GROUP` (grupo macro). La clasificación entre sub-tejidos biológicamente cercanos (corteza frontal vs parietal, ventrículo vs aurícula, adiposo subcutáneo vs visceral) es un problema genuinamente difícil que exige al modelo aprender firmas de expresión génica sutiles — a diferencia de los 5 grupos macro, donde las diferencias transcripcionales son tan marcadas que cualquier clasificador las detecta.

**Preprocesamiento compartido:**  
- `StandardScaler` (sklearn): normaliza a media=0, std=1 — necesario para K-Means (sensible a escala).
- `PCA(50 componentes)` (sklearn): reduce 500 → 50 features, capturando el 82.8% de la varianza. Acelera el entrenamiento y reduce ruido en K-Means.

**Prevención de sobreajuste:**  
- Split por donante (sección 2): evaluación sobre individuos no vistos en entrenamiento.
- RF: `featureSubsetStrategy='sqrt'` introduce aleatoriedad por árbol.
- RF: `maxDepth=10` limita la profundidad máxima.
- PCA antes de K-Means: reduce ruido en features de baja varianza.

In [13]:
# --- Preprocesamiento: StandardScaler + PCA (sklearn, sobre driver) ---
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as SklearnPCA

N_PCA = 50

scaler = StandardScaler(with_mean=True, with_std=True)
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

pca = SklearnPCA(n_components=N_PCA, random_state=RANDOM_SEED)
X_train_pca = pca.fit_transform(X_train_sc)
X_test_pca  = pca.transform(X_test_sc)

cum_var = pca.explained_variance_ratio_.cumsum()
print(f'Varianza explicada acumulada:')
for k in [5, 10, 20, 50]:
    print(f'  PC 1–{k:>2}: {cum_var[k-1]*100:.1f}%')
print(f'\nX_train_pca: {X_train_pca.shape}')
print(f'X_test_pca : {X_test_pca.shape}')

Varianza explicada acumulada:
  PC 1– 5: 37.7%
  PC 1–10: 53.7%
  PC 1–20: 67.5%
  PC 1–50: 82.8%

X_train_pca: (15620, 50)
X_test_pca : (3996, 50)


In [14]:
# --- Crear Spark DataFrames con features PCA ---
from pyspark.ml.feature import VectorAssembler

pca_cols = [f'pc{i:02d}' for i in range(N_PCA)]

def to_spark_pca(X_pca, y_arr, label_col='label'):
    df = pd.DataFrame(X_pca, columns=pca_cols)
    df[label_col] = y_arr
    sdf = spark.createDataFrame(df)
    assembler = VectorAssembler(inputCols=pca_cols, outputCol='pca_features')
    return assembler.transform(sdf).select('pca_features', label_col).cache()

train_pca_spark = to_spark_pca(X_train_pca, y_train)
test_pca_spark  = to_spark_pca(X_test_pca,  y_test)

# Spark DataFrame con features originales (sin PCA) para RF supervisado
# train_raw_spark se cachea — RF hace múltiples pasadas sobre los datos durante el entrenamiento
def to_spark_raw(X_arr, y_arr):
    df_pd = pd.DataFrame(X_arr, columns=gene_cols)
    df_pd['label'] = y_arr
    return spark.createDataFrame(df_pd)

train_raw_spark = to_spark_raw(X_train, y_train).cache()
test_raw_spark  = to_spark_raw(X_test,  y_test)

print(f'train_pca_spark: {train_pca_spark.count():,} filas')
print(f'test_pca_spark : {test_pca_spark.count():,} filas')
print(f'train_raw_spark: {train_raw_spark.count():,} filas')

train_pca_spark: 15,620 filas


test_pca_spark : 3,996 filas


train_raw_spark: 15,620 filas


### 4.2 Modelo supervisado: Random Forest

**Hiperparámetros seleccionados:**

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| `n_estimators` | 100 | Balance entre estabilidad de predicción y tiempo de cómputo |
| `max_depth` | 10 | Limita sobreajuste; suficiente para 54 clases en espacio PCA |
| `max_features` | `sqrt` | Estándar para clasificación; √50 ≈ 7 PCs/árbol |
| `random_state` | 42 | Reproducibilidad |

**Implementación: sklearn en driver (no PySpark MLlib)**  
Con 54 clases SMTSD, el `DTStatsAggregator` de PySpark MLlib (`numClasses × numBins × numFeatures` por partición) agota el heap de la JVM en modo local en Windows, incluso con 50 PCs y 8 GB asignados. Esta es la misma limitación documentada en Tarea 4 para `StandardScaler` + `PCA`: la solución es ejecutar el preprocesamiento y el modelo supervisado en sklearn (sobre el driver), mientras PySpark MLlib maneja el clustering no supervisado (K-Means) y el procesamiento distribuido de metadatos.

**Features de entrada: PCA(50)**  
RF se entrena sobre `X_train_pca` / `X_test_pca` — las mismas 50 componentes PCA usadas en K-Means. PCA(50) captura el 82.8% de la varianza genómica.

**Clasificación por SMTSD — 54 sub-tipos de tejido:**  
A diferencia de los 5 grupos macro (`TISSUE_GROUP`), aquí el modelo debe distinguir sub-tipos biológicamente cercanos. La accuracy esperada es notablemente menor, produciendo un modelo más informativamente útil.

In [ ]:
from sklearn.ensemble import RandomForestClassifier as SklearnRF

# sklearn RF sobre PCA(50) — evita OOM de PySpark MLlib con 54 clases SMTSD en Windows
rf_sk = SklearnRF(
    n_estimators=100, max_depth=10,
    max_features='sqrt', random_state=RANDOM_SEED, n_jobs=-1
)

print(f'Entrenando Random Forest (SMTSD, {len(le_tissue.classes_)} clases, {N_PCA} PCs)...')
rf_sk.fit(X_train_pca, y_train)
print('Entrenamiento completado.')

In [ ]:
# --- Evaluación del modelo supervisado ---
y_pred_rf = rf_sk.predict(X_test_pca)

acc_rf = accuracy_score(y_test, y_pred_rf)
f1_rf  = sk_f1(y_test, y_pred_rf, average='macro', zero_division=0)

print(f'Accuracy (macro): {acc_rf:.4f}')
print(f'F1-Score (macro): {f1_rf:.4f}')
print()
print('Reporte por sub-tipo de tejido (SMTSD):')
print(classification_report(
    y_test, y_pred_rf,
    target_names=le_tissue.classes_,
    zero_division=0
))

In [ ]:
# --- Errores más frecuentes (top-10 pares de confusión) ---
from collections import Counter

label_to_name = dict(enumerate(le_tissue.classes_))
errors_mask = y_pred_rf != y_test
errors_real = y_test[errors_mask]
errors_pred = y_pred_rf[errors_mask]

confusion_pairs = Counter(zip(
    [label_to_name[l] for l in errors_real],
    [label_to_name[l] for l in errors_pred]
))

print(f'Total errores: {errors_mask.sum():,} / {len(y_test):,} muestras')
print()
print('Top 10 pares de confusión más frecuentes:')
print(f'  {"Real":<45} {"Predicho":<45} {"N":>5}')
print('-' * 100)
for (real, pred), cnt in confusion_pairs.most_common(10):
    print(f'  {real:<45} {pred:<45} {cnt:>5}')

In [ ]:
# --- Importancia de componentes PCA (top-20) ---
importances = rf_sk.feature_importances_
feat_imp = sorted(zip(pca_cols, importances), key=lambda x: x[1], reverse=True)

print('Top 20 componentes PCA más importantes para clasificar SMTSD:')
print(f'{"Rank":<6} {"PC":<8} {"Importancia":>12}')
print('-' * 30)
for rank, (col, imp) in enumerate(feat_imp[:20], 1):
    print(f'{rank:<6} {col:<8} {imp:.5f}')

print()
print('Los PCs con mayor importancia capturan las variaciones de expresión génica')
print('más discriminantes entre sub-tipos de tejido, no necesariamente los de')
print('mayor varianza global (PC00, PC01, ...).')

### 4.3 Modelo no supervisado: K-Means con PCA

**Preprocesamiento:** StandardScaler + PCA(50) aplicado en sección 4.1 — el espacio de 50 componentes captura el 82.8% de la varianza genómica sobre las 19,616 muestras.

**Selección de k:** método del codo con índice Silhouette para k=2..7. Con el dataset completo (19,616 muestras, 5 grupos de tejido heterogéneos) se espera que el k óptimo refleje la diversidad intra-grupo: Visceral_Metabólico (7,785 muestras) agrupa hígado, páncreas, pulmón, riñón y otros órganos con perfiles de expresión génica distintos, por lo que el k óptimo puede superar el número de grupos de tejido.

**Hiperparámetros K-Means:**

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| `initMode` | K-Means++ (PySpark default) | Inicialización determinista que reduce iteraciones hasta convergencia |
| `maxIter` | 50 | Suficiente para convergencia en 50 componentes PCA |
| `seed` | 42 | Reproducibilidad |

In [19]:
from pyspark.ml.clustering import KMeans

silhouette_scores = {}
wcss_scores       = {}

print('Método del codo (Silhouette y WCSS para k=2..7):')
print(f'{"k":<6} {"Silhouette":>12} {"WCSS (train)":>15}')
print('-' * 38)

for k in range(2, 8):
    km = KMeans(
        featuresCol='pca_features', predictionCol='prediction',
        k=k, maxIter=50, seed=RANDOM_SEED
    )
    km_model = km.fit(train_pca_spark)
    wcss = km_model.summary.trainingCost

    preds_pd = km_model.transform(test_pca_spark) \
        .select('pca_features', 'prediction').toPandas()
    y_pred = preds_pd['prediction'].values
    X_eval = np.vstack([v.toArray() for v in preds_pd['pca_features']])

    if len(np.unique(y_pred)) < 2:
        sil = -1.0
    else:
        sil = sk_silhouette(X_eval, y_pred, metric='euclidean')

    silhouette_scores[k] = sil
    wcss_scores[k]       = wcss
    print(f'{k:<6} {sil:>12.4f} {wcss:>15,.0f}')

Método del codo (Silhouette y WCSS para k=2..7):
k        Silhouette    WCSS (train)
--------------------------------------


2            0.1646       5,631,143


3            0.1827       5,249,681


4            0.2016       4,876,595


5            0.1771       4,737,735


6            0.2149       4,215,261


7            0.2283       4,002,651


In [20]:
# --- Entrenamiento final con k óptimo ---
k_opt = max(silhouette_scores, key=silhouette_scores.get)
print(f'k óptimo (mayor Silhouette): k={k_opt}  (Silhouette={silhouette_scores[k_opt]:.4f})')

km_final = KMeans(
    featuresCol='pca_features', predictionCol='prediction',
    k=k_opt, maxIter=50, seed=RANDOM_SEED
)
model_km = km_final.fit(train_pca_spark)
print(f'K-Means entrenado con k={k_opt}.')

k óptimo (mayor Silhouette): k=7  (Silhouette=0.2283)


K-Means entrenado con k=7.


In [21]:
# --- Evaluación del clustering: Silhouette + Pureza ---
preds_km_pd = model_km.transform(test_pca_spark) \
    .select('pca_features', 'label', 'prediction').toPandas()

y_km_pred = preds_km_pd['prediction'].values
X_km_eval = np.vstack([v.toArray() for v in preds_km_pd['pca_features']])
y_true    = preds_km_pd['label'].values

sil_final = sk_silhouette(X_km_eval, y_km_pred, metric='euclidean')

# Pureza: fracción de la clase mayoritaria de tejido por cluster
n_correct = 0
print(f'Silhouette (k={k_opt}): {sil_final:.4f}')
print('\nTabla de contingencia cluster vs TISSUE_GROUP:')
ct = pd.crosstab(
    preds_km_pd['prediction'],
    preds_km_pd['label'].map(dict(enumerate(le_tissue.classes_))),
    rownames=['Cluster'],
    colnames=['TISSUE_GROUP']
)
print(ct)
for cluster_id in ct.index:
    n_correct += ct.loc[cluster_id].max()
purity = n_correct / len(preds_km_pd)
print(f'\nPureza del clustering: {purity:.4f} ({n_correct}/{len(preds_km_pd)} muestras correctas)')

Silhouette (k=7): 0.2283

Tabla de contingencia cluster vs TISSUE_GROUP:
TISSUE_GROUP  Cardiovascular  Hematopoyetico  Musculoesqueletico  Nervioso  \
Cluster                                                                      
0                        288              55                 277       135   
1                         31               1                  11       673   
2                          0              60                 132         0   
3                          0               0                 162         0   
4                        154               0                   0         0   
5                          0               0                 283         0   
6                          0             168                   0         0   

TISSUE_GROUP  Visceral_Metabolico  
Cluster                            
0                             819  
1                             543  
2                              55  
3                               0  
4       

In [ ]:
# --- Tabla resumen de resultados ---
print('=' * 60)
print('  RESUMEN DE RESULTADOS — ETAPA 3')
print('=' * 60)
print()
print(f'MODELO SUPERVISADO — Random Forest (SMTSD, {len(le_tissue.classes_)} clases)')
print(f'  Accuracy (macro) : {acc_rf:.4f}')
print(f'  F1-Score (macro) : {f1_rf:.4f}')
print(f'  Split            : por donante (SUBJID), 0 solapamiento')
print()
print('MODELO NO SUPERVISADO — K-Means')
print(f'  k óptimo         : {k_opt}')
print(f'  Silhouette       : {sil_final:.4f}')
print(f'  Pureza           : {purity:.4f}')
print('=' * 60)

---
## 5. Análisis de resultados

### 5.1 Modelo supervisado — Random Forest (SMTSD)

**Resultados obtenidos:** *(ver salida de celda anterior para valores exactos por clase)*

La clasificación por `SMTSD` (~50 sub-tipos de tejido) es un problema biológicamente exigente. A diferencia de los 5 grupos macro (`TISSUE_GROUP`), aquí el modelo debe distinguir entre tejidos cercanos: diferentes regiones cerebrales, distintos tipos de arteria, regiones del colon, sub-tipos de tejido adiposo. La accuracy esperada es considerablemente menor que con grupos macro, lo que hace el modelo más genuinamente informativo.

**Fortalezas:**

- **Clasificación fina de sub-tejidos:** el modelo aprende firmas de expresión génica que distinguen sub-tipos cercanos. Los genes con mayor importancia (ver sección 4.2) revelan marcadores moleculares específicos de cada sub-tejido.
- **Generalización por donante:** la división por `SUBJID` garantiza que las métricas reflejan capacidad de predicción en individuos nuevos — condición necesaria para aplicaciones de medicina espacial.
- **Errores biológicamente interpretables:** las confusiones entre sub-tipos (visible en la matriz de confusión) revelan qué tejidos comparten perfiles transcripcionales similares, información valiosa para entender la organización del transcriptoma humano.

**Áreas de oportunidad:**

- **Desbalance de clases marcado:** sub-tipos de cerebro tienen cientos de muestras mientras que tejidos raros pueden tener <50. Técnicas de balanceo (class weights, oversampling) o F1-macro como métrica de optimización mejorarían la detección de clases minoritarias.
- **Ajuste de hiperparámetros:** `numTrees=100, maxDepth=10` son valores heurísticos. Un grid search con `GroupKFold` por donante podría identificar configuraciones óptimas para ~50 clases.
- **Features adicionales:** incorporar variables demográficas (edad, sexo, IMC) como features podría mejorar la discriminación en sub-tejidos con alta variabilidad inter-individual.

### 5.2 Modelo no supervisado — K-Means

**Resultados obtenidos:** *(ver salida de celda de evaluación para valores exactos)*

**Fortalezas:**

- **Descubrimiento sin etiquetas:** el clustering revela estructura latente en el espacio transcripcional sin usar las etiquetas de sub-tejido. Los clusters reflejan similitud en perfiles de expresión génica, no categorías impuestas.
- **k óptimo > 5:** con el dataset completo (19,616 muestras), la estructura interna excede los grupos macro. El k óptimo refleja sub-grupos dentro de tejidos heterogéneos como Visceral_Metabólico (hígado, páncreas, pulmón, riñón con perfiles distintos).

**Áreas de oportunidad:**

- **Silhouette moderado:** el espacio de 50 PCs captura 82.8% de la varianza, pero la dispersión intra-grupo en 19,616 muestras es alta. UMAP o t-SNE antes del clustering podría mejorar la cohesión.
- **K-Means asume clusters esféricos:** la geometría del transcriptoma en espacio PCA es elíptica. GMM modelaría mejor la forma real de los clusters, a mayor costo computacional.

### 5.3 Síntesis e implicaciones para medicina espacial

Los dos modelos son complementarios y abordan el problema de identificación tisular desde perspectivas distintas:

- El **modelo supervisado (RF)** establece una línea base del transcriptoma humano normal por sub-tipo de tejido. Para medicina espacial, permite detectar si una muestra de un astronauta post-vuelo muestra un perfil de expresión génica que ya no corresponde a su tejido esperado — indicando una alteración molecular inducida por microgravedad o radiación.
- El **modelo no supervisado (K-Means)** descubre estructura latente sin asumir categorías previas. Para medicina espacial, permite detectar si muestras de astronautas forman clusters anómalos que no aparecen en la población GTEx — indicando estados fisiológicos sin precedente en condiciones terrestres.

La clasificación por `SMTSD` en lugar de `TISSUE_GROUP` produce modelos más útiles: los errores del clasificador revelan qué sub-tejidos comparten firmas moleculares, y la detección de anomalías en medicina espacial requiere precisión a nivel de sub-tejido para localizar el origen de los cambios fisiológicos.

---
## Referencias

1. Breiman, L. (2001). Random Forests. *Machine Learning*, 45(1), 5–32. https://doi.org/10.1023/A:1010933404324
2. GTEx Consortium. (2020). The GTEx Consortium atlas of genetic regulatory effects across human tissues. *Science*. https://doi.org/10.1126/science.aaz1776
3. GTEx Portal. (2025). GTEx Analysis V10 Downloads. Broad Institute. https://gtexportal.org/home/downloads/adult-gtex
4. Law, C. W., et al. (2016). voom: Precision weights unlock linear model analysis tools for RNA-seq read counts. *Genome Biology*, 17, 29.
5. Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR*, 12, 2825–2830.
6. Zaharia, M., et al. (2016). Apache Spark: A Unified Engine for Big Data Processing. *Communications of the ACM*, 59(11), 56–65.